# Step 1 - Prototype 2

We now have our first working prototype for step 1. In the second (and probably final) one, we mainly want to fix some smaller issues and decide on foundational formulations / implementations that will make extendability much easier later on. These are:
- Think of a good way to encode and implement train stations (probably already along with the corresponding constraints (essentially blocking movement for some time)).
- Adjust the optimization goal to minimize lateness (for now only at the destination station). Additionally, maybe make sure that the formulation can be extended in terms of travel and dwell time minimization goals later.

Adding a safety distance now might also be doable, however, this is in a sense then also connected to different train speeds (that might require different safety distances) and this is probably really then a point for step 2...

In [ ]:
import gurobipy as gp
from gurobipy import GRB

import numpy as np

In [ ]:
# Initialize the Model
model = gp.Model("Basic_MILP")

### Define the track system and timetable

Here we want to see, if it makes sense to create (separate?) definitions for train stations and block segments that can then hopefully be combined automatically or, if just adding segments of length one with a given 'station capacity' should work just fine (or even better xD).

Given the fact, that we probably only want to have leave and arrival times at stations and not on 'random' points on the tracks, a more native way of distinguishing (separate blocks and train stations) might be beneficial. This could then be handled via an additional boolean array `is_train_station_array` (or something like that xD).

In [ ]:
# Track system data (store number of blocks b_i per segment of tracks with capacities c_i)
station_positions = []
station_capacities = []

segment_lengths = [5, 5, 2]
segment_capacities = [2, 1, 3]

# Timetable data
time_horizon = 20

leave_times = [2, 3, 5]    # Due to the implementation ('looking-back') leave_times_i has to be >= 1
arrival_times = [18, 19, 20]

# Deriving some additional variables
num_trains = len(leave_times)
num_blocks = sum(segment_lengths)

# Helper list for capacity references
capacities_list = np.repeat(segment_capacities, segment_lengths)